# Tarea 3 - Ciencia de Datos
#### Gustavo Hernández Angeles

In [84]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from pathlib import Path
import pathlib
# Aprendizaje No Supervisado
from sklearn.cluster import (
    AgglomerativeClustering,
    DBSCAN,
    KMeans,
    MeanShift,
    SpectralClustering
)
from sklearn.decomposition import (
    KernelPCA
)
# Métricas
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    confusion_matrix,
    silhouette_score
)
import utils.utils as u
import cv2
import os
from typing import Literal

from sklearn.feature_extraction.text import CountVectorizer

## Leer el conjunto de datos

In [31]:
def extraeImagenTexto(path_images, path_labels, size, seed):    
    # Obtenemos imagenes aleatoriamente.
    np.random.seed(seed)    
    image_names = [file.name for file in path_images.glob("*.jpg")]
    image_names = np.random.choice(image_names,size=size,replace=False)
    
    # Leemos el df con etiquetas
    df = pd.read_csv(path_labels, sep="|")
    df.rename(columns=lambda x: x.strip(), inplace=True)

    # Obtenemos los textos de las imagenes.
    image_texts = []
    df_textos = df.groupby("image_name")["comment"].sum()
    for image_name in image_names:
        image_texts.append(df_textos.loc[image_name])
    
    return image_names, image_texts

In [37]:
dir_images = Path("../data/flickr30k_images/")
file_labels = Path("../data/results.csv")

image_names, image_texts = extraeImagenTexto(dir_images, file_labels, size=1000,
                                             seed=1825)

n_images = len(image_names)

print(f"Imagenes leidas: {n_images}")
print(image_names[:2])
print(image_texts[:2])

Imagenes leidas: 1000
['6064872574.jpg' '1538738682.jpg']
[' An old woman seated and wearing glasses making necklaces out of beads under the light of a fluorescent bulb . A woman wearing eyeglasses , sitting in front of the table while making jewelries . A brown-haired woman wearing spectacles crafts jewelery by hand using a desk lamp . A woman is sitting by a desk crafting necklaces out of assorted beads by hand . A middle-aged woman assembles jewelry .', ' A child is sleeping in the front passenger seat of a car with the window slightly open . A child sleeps viewed through a window of a car which is cracked slightly . A young child is sleeping inside a vehicle that is parked outside . A child asleep in a car with the window open A child is sleeping in a car .']


## Representación de Imagenes

In [58]:
def image2Hist(dir_images, image_names, type : Literal["gris","color","combinado"]):
    dim_por_representacion = {
        "gris" : 256,
        "color" : 768,
        "combinado" : 256+768
    }

    if type not in ["gris", "color", "combinado"]:
        raise ValueError('Type debe ser uno de ("gris","color","combinado").')
    
    X = np.zeros((n_images, dim_por_representacion[type]))
    
    
    for i in range(X.shape[0]):
        image_path = os.path.join(dir_images, image_names[i])
        img = cv2.imread(image_path)
        
        if type == "combinado":
            X[i] = np.concatenate([u.obtener_histograma_gris(img),u.obtener_histograma_color(img)])
        elif type == "gris":
            X[i] = u.obtener_histograma_gris(img)
        else:
            X[i] = u.obtener_histograma_color(img)
    
    return X
    

In [62]:
imgPorIntensidad = image2Hist(dir_images, image_names, type="gris")
imgPorIntensidad.shape

(1000, 256)

In [63]:
imgPorColor = image2Hist(dir_images, image_names, type="color")
imgPorColor.shape

(1000, 768)

In [64]:
imgCombinado = image2Hist(dir_images, image_names, type="combinado")
imgCombinado.shape

(1000, 1024)

## Representación de Textos

In [87]:
vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(image_texts)
vectorizer.get_feature_names_out()

array(['08', '10', '12', ..., 'zoo', 'zoom', 'zooms'], dtype=object)

## Algoritmos de Agrupamiento

## Evaluación de Resultados